In [ ]:
import pandas as pd
import matplotlib
import ast
#matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore')

#폰트 설정(없으면 영문으로 나옴)
try:
    font_path = [f for f in fm.findSystemFonts() if 'NanumGothic' in f or 'Malgun' in f or 'AppleGothic' in f]
    if font_path:
        plt.rcParams['font.family'] = fm.FontProperties(fname=font_path[0]).get_name()
    else:
        plt.rcParams['font.family'] = 'DejaVu Sans'
except:
    plt.rcParams['font.family'] = 'DejaVu Sans'

plt.rcParams['axes.unicode_minus'] = False



#--------------------
# 데이터 로드 함수
#--------------------   

def load_data(file_path):
    try:
        df = pd.read_csv(file_path)
        df.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')
        return df
    except Exception as e:
        print(f"Error loading data: {e}")
        return None

portfolio = load_data('../dataset/portfolio.csv')
profile = load_data('../dataset/profile.csv')
transcript = load_data('../dataset/transcript.csv')

portfolio.head()


#--------------------
# 결측치 채우기
#--------------------

def fill_unknown(df, columns):
    for col in columns:
        df[col] = df[col].fillna('Unknown')
    return df

profile = fill_unknown(profile, ['gender', 'income'])



#-----------------------
# 프로필 결측 플래그 생성
#-----------------------

# income, gender 결측이 2,175명으로 동일
# age=118도 동일 개수 → 가입 시 정보 미입력 고객군으로 판단
# 단순 이상치 제거 대신 명시적 플래그로 관리

profile['is_profile_missing'] = (
    (profile['gender'] == 'Unknown') |
    (profile['income'] == 'Unknown') |
    (profile['age'] == 118)
).astype(int)

print(f"프로필 미입력 고객 (Unknown 처리된 고객): {profile['is_profile_missing'].sum():,}명")
print(f"비율: {profile['is_profile_missing'].mean()*100:.1f}%")



# ----------------------
# portfolio 전처리
# ----------------------

# channels 리스트 문자열 → 이진 4컬럼
portfolio['channels_list'] = portfolio['channels'].apply(ast.literal_eval)

for ch in ['web', 'email', 'mobile', 'social']:
    portfolio[f'ch_{ch}'] = portfolio['channels_list'].apply(
        lambda x: 1 if ch in x else 0
    )

# 채널 수 (몇 개 채널로 발송됐는지)
portfolio['channel_count'] = (
    portfolio['ch_web'] + portfolio['ch_email'] +
    portfolio['ch_mobile'] + portfolio['ch_social']
)

portfolio.drop(columns=['channels', 'channels_list'], inplace=True)

print("✅ portfolio 전처리 완료")
portfolio.head()



#-----------------------
# profile 전처리
#-----------------------

# age == 118 → 'Unknown' 마스킹
profile['age'] = profile['age'].where(profile['age'] != 118, other='Unknown')

# became_member_on → datetime 변경
profile['became_member_on'] = pd.to_datetime(
    profile['became_member_on'].astype(str), format='%Y%m%d'
)

# 멤버십 기간(일수): 데이터 기준일 = 가입일 최댓값 + 1일
ref_date = profile['became_member_on'].max() + pd.Timedelta(days=1)
profile['member_days'] = (ref_date - profile['became_member_on']).dt.days

# 연령대 × 성별 파생변수
def group_age_gender(df):
    if df['gender'] == 'Unknown' or df['age'] == 'Unknown':
        return '미기입'
    elif df['gender'] == 'O':
        return 'Others'
    elif df['gender'] == 'M':
        if df['age'] < 20:   return '20세 미만 남성'
        elif df['age'] < 30: return '20대 남성'
        elif df['age'] < 40: return '30대 남성'
        elif df['age'] < 50: return '40대 남성'
        elif df['age'] < 60: return '50대 남성'
        elif df['age'] < 70: return '60대 남성'
        else:                return '60+ 남성'
    else:
        if df['age'] < 20:   return '20세 미만 여성'
        elif df['age'] < 30: return '20대 여성'
        elif df['age'] < 40: return '30대 여성'
        elif df['age'] < 50: return '40대 여성'
        elif df['age'] < 60: return '50대 여성'
        elif df['age'] < 70: return '60대 여성'
        else:                return '60+ 여성'

profile['age_gender'] = profile.apply(group_age_gender, axis=1)
profile['age_gender'].value_counts()

# 소득 구간 파생변수
def income_group(income):
    if income == 'Unknown':
        return '누락'
    elif float(income) < 50000:  return '5만 미만'
    elif float(income) < 75000:  return '5-7.5만'
    elif float(income) < 100000: return '7.5-10만'
    else:                        return '10만 이상'

profile['income_group'] = profile['income'].apply(income_group)

# 멤버십 기간 구간 파생변수
def member_period(days):
    if pd.isna(days) or days < 0: return '미기재'
    elif days < 365:              return '신규 (1년 미만)'
    elif days < 730:              return '중간 (1~2년)'
    else:                         return '장기 (2년+)'

profile['member_days_group'] = profile['member_days'].apply(member_period)



#-----------------------
# transcript 전처리
#-----------------------
# 주의: received/viewed → 'offer id' (공백)
#       completed → 'offer_id'  (언더스코어)  ← key 이름 다름!

def split_value(log):
    d = ast.literal_eval(log['value'])#literal_eval : 문자열로 된 파이썬 자료형을 실제 자료형으로 변환
    if log['event'] == 'transaction':
        return pd.Series({
            'amount': d.get('amount'),
            'offer_id': None
        })
    else:
        # 두 가지 key 이름 모두 처리
        offer_id = d.get('offer id') or d.get('offer_id')
        return pd.Series({
            'amount': None,
            'offer_id': offer_id
        })

parsed = transcript.apply(split_value, axis=1)
transcript = pd.concat([transcript, parsed], axis=1)
transcript.drop(columns=['value'], inplace=True)

#time-> day
transcript['day'] = transcript['time'] // 24

transcript.head()




#-----------------------
# viewed 없이 completed만 있는 경우
#-----------------------

viewed_set = set(
    transcript[transcript['event'] == 'offer viewed']
    [['person', 'offer_id']]
    .apply(tuple, axis=1)
)

# completed 이벤트에 대해, 해당 person과 offer_id가 viewed_set에 있는지 확인
def viewed_flag(log):
    if log['event'] != 'offer completed':
        return None
    return 1 if (log['person'], log['offer_id']) in viewed_set else 0

transcript['viewed_before_complete'] = transcript.apply(viewed_flag, axis=1)


unaware = transcript[transcript['viewed_before_complete'] == 0].shape[0]
total_completed = transcript[transcript['event'] == 'offer completed'].shape[0]
print(f" 이벤트 미인지 완료: {unaware}건 / 전체 완료: {total_completed}건 ({unaware/total_completed*100:.1f}%)")


# ----------------------
# 5. 데이터 병합
# ----------------------

#transcript + profile 병합 (person → id)
full_data = transcript.merge(
    profile.rename(columns={'id': 'person'}),
    on='person', 
    how='left')

#full_data + portfolio 병합 (offer_id → id)
full_data = full_data.merge(
    portfolio.rename(columns={'id': 'offer_id'}),
    on='offer_id',
    how='left')

print("✅ 데이터 병합 완료")
print(f"병합된 데이터 크기: {full_data.shape}")
full_data.head()

